In [ ]:
# Notebook overview: 
# This notebook builds the trainer architecture module used in simulator_final
# The trainer architecture is made of 3 blocks, designed to train the simulator using a 
# a hybrid of classical ML (ADAM) and quantum ML (shot-based gradient estimation)

import numpy as np
class HybridModel:
    def __init__(self, classical_pre, circuit, H_head):
        self.classical_pre = classical_pre
        self.circuit = circuit
        self.H_head = H_head

    def forward(self, x, theta):
        z = self.classical_pre(x)
        self.circuit.run(theta)
        q_feat = self.circuit.expectation(self.H_head)
        return q_feat, z


class SimpleAdam:
    def __init__(self, lr=1e-2):
        self.lr = lr
        self.m = None
        self.v = None
        self.t = 0

    def update(self, theta, grad):
        if self.m is None:
            self.m = np.zeros_like(theta)
            self.v = np.zeros_like(theta)
        self.t += 1
        self.m = 0.9*self.m + 0.1*grad
        self.v = 0.999*self.v + 0.001*(grad**2)
        m_hat = self.m / (1 - 0.9**self.t)
        v_hat = self.v / (1 - 0.999**self.t)
        return theta - self.lr * m_hat / (np.sqrt(v_hat) + 1e-8)


class Trainer:
    def __init__(self, hybrid_model, grad_estimator, optimizer, data_loader):
        self.model = hybrid_model
        self.grad_estimator = grad_estimator
        self.optimizer = optimizer
        self.data_loader = data_loader

    def step(self, theta):
        batch_loss = 0.0
        for x, y in self.data_loader:
            q_feat, _ = self.model.forward(x, theta)
            loss = 0.5 * (q_feat - y)**2
            batch_loss += loss
        batch_loss /= len(self.data_loader)

        grads = np.zeros_like(theta)
        shift = np.pi / 2.0
        for idx in range(len(theta)):
            grads[idx] = self.grad_estimator.estimate_gradient(theta, idx, shift)

        theta_new = self.optimizer.update(theta, grads)
        return theta_new, batch_loss

# For example usage, see trainer_api.ipynb